# SDET Playwright Test Generator — Kaggle Training

Fine-tune Qwen3-8B on SDET Playwright test generation conversations.

**Prerequisites:**
1. Upload `training_unsloth.jsonl` as a Kaggle Dataset named `training-unsloth` (or adjust the path below)
2. Enable **Internet** access in Notebook Settings (top-right panel)
3. Use **GPU T4 x2** or **GPU P100** accelerator

**Output:** LoRA adapter saved to `/kaggle/working/qwen3-8b-sdet/`

In [ ]:
# === Cell 1: Install dependencies ===
# Use --no-deps to avoid backtracking hell with Kaggle's pre-installed torch
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
major_version, minor_version = torch.__version__.split(".")[:2]
major_version, minor_version = int(major_version), int(minor_version)

unsloth_pkg = "unsloth[cu124-torch250]" if (major_version >= 2 and minor_version >= 5) else "unsloth[cu121-torch240]"
!pip install "{unsloth_pkg} @ git+https://github.com/unslothai/unsloth.git" --no-deps -q
!pip install unsloth_zoo bitsandbytes xformers trl peft accelerate wandb datasets -q

# For inference: Playwright browser engine
!pip install playwright -q
!playwright install chromium 2>&1 | tail -3

In [ ]:
# === Cell 2: Copy the inference module into working dir ===
# This makes sdet-inference available as "python -m sdet_inference.cli"
import os, json

os.makedirs("/kaggle/working/sdet_inference", exist_ok=True)

# Embed the inference module directly — no external package dependency needed on Kaggle
# (These files define PageScraper, SDETInference, format_page_context, and the CLI)

PAGE_SCRAPER_PY = '''from __future__ import annotations

import asyncio
import json
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional

@dataclass
class InteractiveElement:
    tag: str
    type: Optional[str] = None
    label: Optional[str] = None
    id: Optional[str] = None
    name: Optional[str] = None
    role: Optional[str] = None
    placeholder: Optional[str] = None
    text: Optional[str] = None
    href: Optional[str] = None
    aria_label: Optional[str] = None

@dataclass
class PageSnapshot:
    url: str
    title: str
    elements: List[InteractiveElement] = field(default_factory=list)
    a11y_tree: Optional[Dict[str, Any]] = None
    text_content: Optional[str] = None
    viewport: Optional[Dict[str, int]] = None

    def to_dict(self) -> Dict[str, Any]:
        return {
            "url": self.url,
            "title": self.title,
            "elements": [asdict(e) for e in self.elements],
            "a11y_tree": self.a11y_tree,
            "viewport": self.viewport,
        }

_INTERACTIVE_SELECTOR = ", ".join([
    "button", "input", "select", "textarea",
    'a[href]', '[role="button"]', '[role="link"]',
    '[role="checkbox"]', '[role="radio"]', '[role="tab"]',
    '[role="combobox"]', '[role="listbox"]', '[role="option"]',
    '[role="menuitem"]', '[role="slider"]', '[role="switch"]',
    '[role="textbox"]', '[role="searchbox"]', '[role="spinbutton"]',
    '[role="treeitem"]', '[contenteditable="true"]',
])

_EXTRACT_ELEMENTS_JS = f"""
() => {{
    const seen = new Set();
    return [...document.querySelectorAll('{_INTERACTIVE_SELECTOR}')].filter(el => {{
        const rect = el.getBoundingClientRect();
        const style = getComputedStyle(el);
        if (rect.width === 0 || rect.height === 0) return false;
        if (style.display === 'none' || style.visibility === 'hidden') return false;
        return true;
    }}).map(el => {{
        const key = el.tagName + (el.id || '') + (el.name || '') + (el.textContent || '').trim().slice(0, 30);
        if (seen.has(key)) return null;
        seen.add(key);
        const label = el.labels?.[0]?.textContent?.trim()
            || el.getAttribute('aria-label')
            || el.placeholder
            || el.textContent?.trim().slice(0, 80)
            || '';
        return {{
            tag: el.tagName,
            type: el.type || null,
            label: label.slice(0, 120),
            id: el.id || null,
            name: el.getAttribute('name') || null,
            role: el.getAttribute('role') || el.type || el.tagName,
            placeholder: el.placeholder || null,
            text: (el.textContent || '').trim().slice(0, 80) || null,
            href: el.getAttribute('href') || null,
            aria_label: el.getAttribute('aria-label') || null,
        }};
    }}).filter(Boolean);
}}
"""

_PAGE_TEXT_JS = """() => {
    const main = document.querySelector('main, [role="main"], article, .content, #content');
    const target = main || document.body;
    const clone = target.cloneNode(true);
    clone.querySelectorAll('script, style, nav, footer, header, aside, .sidebar').forEach(el => el.remove());
    return (clone.textContent || '').replace(/\\s+/g, ' ').trim().slice(0, 5000);
}"""

async def extract_interactive_elements(page: Any) -> List[Dict[str, Any]]:
    return await page.evaluate(_EXTRACT_ELEMENTS_JS)

async def extract_a11y_tree(page: Any) -> Optional[Dict[str, Any]]:
    try:
        return await page.accessibility.snapshot()
    except Exception:
        return None

class PageScraper:
    def __init__(self, headless: bool = True, viewport: Optional[Dict[str, int]] = None, timeout_ms: int = 30000):
        self._headless = headless
        self._viewport = viewport or {"width": 1280, "height": 720}
        self._timeout_ms = timeout_ms
        self._browser = None

    async def __aenter__(self):
        from playwright.async_api import async_playwright
        self._pw = await async_playwright().start()
        self._browser = await self._pw.chromium.launch(headless=self._headless)
        return self

    async def __aexit__(self, *args):
        if self._browser:
            await self._browser.close()
        await self._pw.stop()

    async def scrape(self, url: str, wait_until: str = "networkidle", extra_wait_ms: Optional[int] = None, auth_cookies: Optional[List[Dict[str, Any]]] = None) -> PageSnapshot:
        if not self._browser:
            raise RuntimeError("use 'async with PageScraper() as scraper:'")
        context = await self._browser.new_context(viewport=self._viewport)
        if auth_cookies:
            await context.add_cookies(auth_cookies)
        page = await context.new_page()
        page.set_default_timeout(self._timeout_ms)
        try:
            await page.goto(url, wait_until=wait_until, timeout=self._timeout_ms)
            if extra_wait_ms:
                await asyncio.sleep(extra_wait_ms / 1000)
            title = await page.title()
            elements = await extract_interactive_elements(page)
            a11y_tree = await extract_a11y_tree(page)
            text_content = await page.evaluate(_PAGE_TEXT_JS)
            return PageSnapshot(
                url=url, title=title,
                elements=[InteractiveElement(**el) for el in elements],
                a11y_tree=a11y_tree,
                text_content=text_content[:5000] if text_content else None,
                viewport=self._viewport,
            )
        finally:
            await context.close()
'''

with open("/kaggle/working/sdet_inference/page_scraper.py", "w") as f:
    f.write(PAGE_SCRAPER_PY)

print("page_scraper.py written")

# Write inference.py
INFERENCE_PY = '''from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

from sdet_inference.page_scraper import PageSnapshot, InteractiveElement

def _fmt_elements(elements: List[InteractiveElement]) -> str:
    lines = []
    for el in elements:
        parts = [f'<{el.tag.lower()}>']
        if el.label:
            parts.append(f'label="{el.label}"')
        if el.role:
            parts.append(f"role={el.role}")
        if el.type and el.type != "text":
            parts.append(f"type={el.type}")
        if el.id:
            parts.append(f"id=#{el.id}")
        if el.name:
            parts.append(f"name={el.name}")
        if el.href:
            parts.append(f"href={el.href[:60]}")
        lines.append("  " + " ".join(parts))
    return "\\n".join(lines)

def format_page_context(snapshot: PageSnapshot) -> str:
    elements_text = _fmt_elements(snapshot.elements)
    a11y_text = json.dumps(snapshot.a11y_tree, indent=2)[:2000] if snapshot.a11y_tree else "N/A"
    return f"""=== Page Context ===
URL: {snapshot.url}
Title: {snapshot.title}

=== Interactive Elements ===
{elements_text}

=== Accessibility Tree ===
{a11y_text}
"""

@dataclass
class InferenceConfig:
    model_path: str
    base_model_name: str = "Qwen/Qwen3-8B"
    max_seq_length: int = 8192
    max_new_tokens: int = 2048
    temperature: float = 0.7
    top_p: float = 0.9
    load_in_4bit: bool = True
    use_lora: bool = True

class SDETInference:
    def __init__(self, config: InferenceConfig):
        self._config = config
        self._model = None
        self._tokenizer = None

    def load(self):
        from unsloth import FastLanguageModel
        if self._config.use_lora and os.path.isdir(self._config.model_path):
            from peft import PeftModel
            base, tokenizer = FastLanguageModel.from_pretrained(
                model_name=self._config.base_model_name,
                max_seq_length=self._config.max_seq_length,
                dtype=None, load_in_4bit=self._config.load_in_4bit,
            )
            model = PeftModel.from_pretrained(base, self._config.model_path)
        else:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=self._config.model_path,
                max_seq_length=self._config.max_seq_length,
                dtype=None, load_in_4bit=self._config.load_in_4bit,
            )
        FastLanguageModel.for_inference(model)
        self._model = model
        self._tokenizer = tokenizer

    def generate(self, scenario: str, page_snapshot: Optional[PageSnapshot] = None, system_prompt: Optional[str] = None) -> str:
        if not self._model or not self._tokenizer:
            raise RuntimeError("call .load() before .generate()")
        parts = []
        if system_prompt:
            parts.append(f"System: {system_prompt}")
        if page_snapshot:
            parts.append(format_page_context(page_snapshot))
        parts.append(f"=== Task ===\\n{scenario}")
        parts.append(
            "Follow the SDET workflow: analyze the feature, identify elements and locators, "
            "plan the action sequence with assertions, apply reliability hardening, "
            "then produce the complete Playwright test code."
        )
        user_content = "\\n\\n".join(parts)
        messages = [{"role": "user", "content": user_content}]
        text = self._tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self._tokenizer([text], return_tensors="pt", padding=True)
        inputs = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in inputs.items()}
        outputs = self._model.generate(
            **inputs,
            max_new_tokens=self._config.max_new_tokens,
            temperature=self._config.temperature,
            top_p=self._config.top_p,
            do_sample=True,
        )
        full_output = self._tokenizer.decode(outputs[0], skip_special_tokens=False)
        assistant_tag = "<|im_start|>assistant"
        if assistant_tag in full_output:
            return full_output.split(assistant_tag)[-1].split("<|im_end|>")[0].strip()
        return full_output.strip()

    @property
    def is_loaded(self) -> bool:
        return self._model is not None
'''

with open("/kaggle/working/sdet_inference/inference.py", "w") as f:
    f.write(INFERENCE_PY)

# Write CLI
CLI_PY = '''from __future__ import annotations

import argparse
import asyncio
import json
import sys

def main():
    parser = argparse.ArgumentParser(description="SDET Playwright Test Generator")
    sub = parser.add_subparsers(dest="command", required=True)

    scrape_p = sub.add_parser("scrape", help="Scrape a page and save snapshot")
    scrape_p.add_argument("url")
    scrape_p.add_argument("--output", "-o", default="page_snapshot.json")
    scrape_p.add_argument("--wait", type=int, default=30)
    scrape_p.add_argument("--viewport", default="1280x720")
    scrape_p.add_argument("--auth-cookies")

    gen_p = sub.add_parser("generate", help="Generate test from page snapshot")
    gen_p.add_argument("scenario")
    gen_p.add_argument("--snapshot", "-s", default="page_snapshot.json")
    gen_p.add_argument("--model", "-m", required=True)
    gen_p.add_argument("--base-model", default="Qwen/Qwen3-8B")
    gen_p.add_argument("--max-tokens", type=int, default=2048)
    gen_p.add_argument("--temperature", type=float, default=0.7)

    run_p = sub.add_parser("run", help="Scrape + generate in one command")
    run_p.add_argument("url")
    run_p.add_argument("scenario")
    run_p.add_argument("--model", "-m", required=True)
    run_p.add_argument("--base-model", default="Qwen/Qwen3-8B")
    run_p.add_argument("--output", "-o")
    run_p.add_argument("--wait", type=int, default=30)
    run_p.add_argument("--max-tokens", type=int, default=2048)
    run_p.add_argument("--temperature", type=float, default=0.7)

    args = parser.parse_args()

    if args.command == "scrape":
        asyncio.run(_scrape(args))
    elif args.command == "generate":
        _generate(args)
    elif args.command == "run":
        asyncio.run(_run(args))

async def _scrape(args):
    from sdet_inference.page_scraper import PageScraper
    parts = [int(x) for x in args.viewport.split("x")]
    vp = {"width": parts[0], "height": parts[1]} if len(parts) == 2 else None
    auth = None
    if args.auth_cookies:
        with open(args.auth_cookies) as f:
            auth = json.load(f)
    async with PageScraper(viewport=vp, timeout_ms=args.wait * 1000) as scraper:
        snap = await scraper.scrape(args.url, auth_cookies=auth)
    with open(args.output, "w") as f:
        json.dump(snap.to_dict(), f, indent=2)
    print(f"Saved: {snap.title} ({len(snap.elements)} elements)")

def _generate(args):
    from sdet_inference.inference import SDETInference, InferenceConfig
    from sdet_inference.page_scraper import PageSnapshot, InteractiveElement
    with open(args.snapshot) as f:
        data = json.load(f)
    snap = PageSnapshot(
        url=data["url"], title=data.get("title", ""),
        elements=[InteractiveElement(**el) for el in data.get("elements", [])],
        a11y_tree=data.get("a11y_tree"), viewport=data.get("viewport"),
    )
    config = InferenceConfig(model_path=args.model, base_model_name=args.base_model,
                              max_new_tokens=args.max_tokens, temperature=args.temperature)
    engine = SDETInference(config)
    engine.load()
    print(engine.generate(scenario=args.scenario, page_snapshot=snap))

async def _run(args):
    from sdet_inference.page_scraper import PageScraper
    from sdet_inference.inference import SDETInference, InferenceConfig
    async with PageScraper(timeout_ms=args.wait * 1000) as scraper:
        snap = await scraper.scrape(args.url)
    print(f"Scraped: {snap.title} ({len(snap.elements)} elements)")
    config = InferenceConfig(model_path=args.model, base_model_name=args.base_model,
                              max_new_tokens=args.max_tokens, temperature=args.temperature)
    engine = SDETInference(config)
    engine.load()
    result = engine.generate(scenario=args.scenario, page_snapshot=snap)
    if args.output:
        with open(args.output, "w") as f:
            f.write(result)
    else:
        print(result)

if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/sdet_inference/cli.py", "w") as f:
    f.write(CLI_PY)

with open("/kaggle/working/sdet_inference/__init__.py", "w") as f:
    f.write("from sdet_inference.page_scraper import PageScraper, PageSnapshot, InteractiveElement\\n")
    f.write("from sdet_inference.inference import SDETInference, InferenceConfig, format_page_context\\n")

print("Inference module written to /kaggle/working/sdet_inference/")
print("Use:  python -m sdet_inference.cli run <url> <scenario> --model <path>")

In [ ]:
# === Cell 3: Imports ===
import json, os, glob, torch
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from transformers import TrainingArguments
from trl import SFTTrainer

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} x {torch.cuda.device_count()}")

In [ ]:
# === Cell 4: Load dataset ===
import os, json, glob
from datasets import Dataset

# Try multiple possible Kaggle dataset paths
CANDIDATES = [
    "/kaggle/input/training-unsloth/training_unsloth.jsonl",
    "/kaggle/input/training-unsloth/training_data_bigpickle.jsonl",
    "/kaggle/input/trainingunsloth/training_unsloth.jsonl",
    "/kaggle/input/sdet-training/training_unsloth.jsonl",
]

DATA_PATH = None
for p in CANDIDATES:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    matches = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
    if matches:
        DATA_PATH = matches[0]
        print(f"Auto-discovered dataset: {DATA_PATH}")
    else:
        raise FileNotFoundError(
            "No JSONL dataset found. Upload training_unsloth.jsonl as a Kaggle Dataset."
        )

OUTPUT_DIR = "/kaggle/working/qwen3-8b-sdet"
MODEL_NAME = "Qwen/Qwen3-8B"
# 2048 context fits T4 comfortably; conversations avg ~600 tokens
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-5
NUM_EPOCHS = 1

print(f"Dataset: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

# Load raw ShareGPT conversations
raw = []
with open(DATA_PATH) as f:
    for line in f:
        raw.append(json.loads(line))
print(f"Loaded {len(raw)} conversations")

if len(raw) == 0:
    raise ValueError("Dataset is empty")

# Validate structure
assert "conversations" in raw[0], "Missing conversations key"
print(f"Turns per conversation: {len(raw[0]["conversations"])}")
print(f"Role format: {raw[0]["conversations"][0]["from"]}")

# Convert to simple format for SFTTrainer with chat template
# We need the tokenizer for apply_chat_template, which is loaded in Cell 5
# So we store raw and convert after tokenizer is loaded (Cell 6)
print("Ready — tokenizer will format in Cell 6")


In [ ]:
# === Cell 5: Load Qwen3-8B with 4-bit QLoRA ===
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    quantization_config=quant_config,
    device_map="auto",
    use_cache=False,
)


In [ ]:
# === Cell 6: Convert to text format ===
import torch

# Apply Qwen3 chat template to each conversation → produces "text" column
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-3",
)

def to_text(conversations):
    """Convert ShareGPT conversations to formatted text using chat template."""
    # Map 'from'/'value' → 'role'/'content'
    role_map = {"gpt": "assistant", "human": "user"}
    messages = [
        {"role": role_map.get(m["from"], m["from"]), "content": m["value"]}
        for m in conversations
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

texts = [to_text(item["conversations"]) for item in raw]
dataset = Dataset.from_list([{"text": t} for t in texts])

# Check for empty texts
empty = sum(1 for t in texts if not t.strip())
print(f"Total: {len(texts)}, Empty texts: {empty}")
if empty > 0:
    print("WARNING: empty texts found, filtering them out")
    dataset = dataset.filter(lambda x: x["text"].strip() != "")

# Train/test split
dataset = dataset.train_test_split(test_size=0.01, seed=42)
print(f"Train: {len(dataset["train"])}, Eval: {len(dataset["test"])}")

if len(dataset["train"]) == 0:
    raise ValueError("Training split is empty")

# Show a sample of the formatted text
print(f"\nSample text:\n{dataset["train"]["text"][0][:400]}")


In [ ]:
# === Cell 7: Apply LoRA ===
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# === Cell 8: Training arguments ===
args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=200,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
        output_dir=OUTPUT_DIR,
    report_to="none",
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="no",
    seed=42,
)

In [ ]:
# === Cell 9: SFT Trainer with response-only masking ===
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

# Mask user inputs so loss is only computed on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# === Cell 10: Train! ===
import gc, torch, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU memory before training: {torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]:.0f} MiB used")

# B=2, GA=4, 1 epoch: T4 ~1.2h  |  P100 ~1.5h  |  A100 ~20min
train_stats = trainer.train()
print(f"Training complete. Loss: {train_stats.training_loss:.4f}")

In [ ]:
# === Cell 11: Save LoRA adapter ===
model.save_pretrained_merged(OUTPUT_DIR, tokenizer, save_method="lora")
print(f"Model saved to {OUTPUT_DIR}")
!ls -lh {OUTPUT_DIR}

In [ ]:
# === Cell 12: Quick inference test ===
FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "Write a Playwright test for the login page at https://app.example.com/login"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt", padding=True).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(response[:500])

In [ ]:
# === Cell 13: Run inference via CLI ===
# The sdet_inference module is ready at /kaggle/working/sdet_inference/
# It can scrape any live URL, extract elements + a11y tree,
# and feed them to the trained model for test generation.
#
# Examples:
#
# !python -m sdet_inference.cli scrape https://example.com/login --output snapshot.json
#
# !python -m sdet_inference.cli generate "Write a test for SSO login" \
#     --model /kaggle/working/qwen3-8b-sdet --snapshot snapshot.json
#
# !python -m sdet_inference.cli run https://example.com/login \
#     "Write a test for successful SSO login with remember-me" \
#     --model /kaggle/working/qwen3-8b-sdet

print("Inference CLI ready. Use the command above to generate tests.")